In [ ]:
from molcraft.utils import SMILESTokenizer
from molcraft.utils import Sampler 
from molcraft.models import GPT
from molcraft.utils import metrics
from molcraft.utils import vis

from rdkit import RDLogger
from rdkit import Chem
import tensorflow as tf

import pandas as pd
import numpy as np
import keras
import re
import collections

import warnings 

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*') 

## 1. Read data

In [ ]:
filename = './your_data.txt'

with open(filename, 'r') as fh:
    data = fh.read().splitlines()

np.random.shuffle(data)

## 2. Build TensorFlow datasets

In [ ]:
tokenizer = SMILESTokenizer()

tokenizer.adapt(data)

def preprocess(smiles):
    tokens = tokenizer.tokenize(smiles)
    x = tokens[:, :-1]
    y = tokens[:, 1:]
    return x, y
    
ds = tf.data.Dataset.from_tensor_slices(data)
ds = ds.shuffle(4196)
ds = ds.batch(32)
ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.prefetch(tf.data.AUTOTUNE)

## 3. Build, compile, and train GPT 

In [ ]:
gpt = GPT(
    num_layers=4,
    num_heads=8,
    embedding_dim=128,
    dense_dim=512,
    vocabulary_size=tokenizer.vocabulary_size,
    sequence_length=tokenizer.max_sequence_length,
    dropout=0.2,
)

gpt.compile(
    optimizer=keras.optimizers.Adam(1e-3), 
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

gpt.fit(ds, epochs=5)

## 4. Sample molecules

In [ ]:
sampler = Sampler(gpt, tokenizer, temperature=0.7, top_k=None)

input_sequences = [
   '[BOS]' for _ in range(128)
]

smiles = sampler.sample(input_sequences)
smiles = smiles.numpy().astype(str)

print(f'Novelty   = {metrics.novelty(smiles, data):.3f}')
print(f'Diversity = {metrics.diversity(smiles):.3f}')
print(f'Validity  = {metrics.validity(smiles):.3f}')

## 5. Visualize generated molecules

In [ ]:
vis.visualize_smiles(smiles, grid_size=(5, 5))